# Benchmark: Multi-Writer Insert into `v2_pred_patch`

This notebook benchmarks concurrent (multi-writer) insert speeds into the live `v2_pred_patch` table using multiple threads. Each thread performs a bulk insert of N rows, and timings are measured for each writer and overall throughput.

**Outline:**
1. Import Required Libraries
2. Define Connection and Benchmark Parameters
3. Worker Function for Parallel Inserts
4. Run Multi-Writer Benchmark Using ThreadPool
5. Aggregate and Display Results
6. Cleanup Inserted Rows

In [7]:
# 1. Import Required Libraries
import os
import time
import threading
import concurrent.futures
import psycopg2
import statistics
from os import cpu_count

In [8]:
# 2. Define Connection and Benchmark Parameters
DB_HOST     = os.environ.get('DB_HOST',     'prototyping-pg-1')
DB_NAME     = os.environ.get('DB_NAME',     'testdb')
DB_USER     = os.environ.get('DB_USER',     'testuser')
DB_PASSWORD = os.environ.get('DB_PASSWORD', 'mypassword')
DSN = f'dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} host={DB_HOST}'

N_WRITERS      = cpu_count()        # Number of concurrent writers (threads)
ROWS_PER_WRITER = 1000    # Rows inserted by each writer
SENTINEL_UID   = 999_000_000_000  # Sentinel patch_uid for safe cleanup
TABLE          = 'v2_pred_patch'

In [9]:
# 3. Worker Function for Parallel Inserts
def insert_worker(worker_id, n_rows, sentinel_uid, dsn, table):
    conn = psycopg2.connect(dsn)
    conn.autocommit = False
    cur = conn.cursor()
    cur.execute("SET synchronous_commit = OFF;")
    cur.execute("SET work_mem = '256MB';")
    conn.commit()
    
    insert_sql = f"""
        INSERT INTO {table} (patch_uid, embed_coords, grid_cell_i, grid_cell_j, event_ts, pred_label, patch_coords)
        SELECT
            {sentinel_uid} + {worker_id}*1000000 + gs,
            POINT(random() * 10000, random() * 10000),
            (random() * 9999)::INT,
            (random() * 9999)::INT,
            NOW(),
            (random() * 9)::INT,
            POINT(random() * 10000, random() * 10000)
        FROM generate_series(1, {n_rows}) AS gs
        RETURNING id;
    """
    t_start = time.perf_counter()
    cur.execute(insert_sql)
    inserted_ids = [row[0] for row in cur.fetchall()]
    conn.commit()
    t_end = time.perf_counter()
    conn.close()
    elapsed = t_end - t_start
    return {
        'worker_id': worker_id,
        'elapsed': elapsed,
        'inserted_ids': inserted_ids
    }

In [10]:
# 4. Run Multi-Writer Benchmark Using ThreadPool
all_results = []
start_time = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=N_WRITERS) as executor:
    futures = [
        executor.submit(insert_worker, i, ROWS_PER_WRITER, SENTINEL_UID, DSN, TABLE)
        for i in range(N_WRITERS)
    ]
    for future in concurrent.futures.as_completed(futures):
        all_results.append(future.result())
end_time = time.perf_counter()
total_elapsed = end_time - start_time

# Sort results by worker_id for reporting
all_results.sort(key=lambda r: r['worker_id'])

In [11]:
# 5. Aggregate and Display Results
per_thread_times = [r['elapsed'] for r in all_results]
total_rows = N_WRITERS * ROWS_PER_WRITER
throughput = total_rows / total_elapsed if total_elapsed > 0 else float('nan')

print(f"\n=== Multi-Writer Benchmark Result ===")
print(f"Writers (threads): {N_WRITERS}")
print(f"Rows per writer : {ROWS_PER_WRITER}")
print(f"Total rows      : {total_rows}")
print(f"Total elapsed   : {total_elapsed*1000:.3f} ms  ({total_elapsed:.6f} s)")
print(f"Overall throughput: {throughput:,.0f} rows/sec\n")

for r in all_results:
    print(f"Worker {r['worker_id']}: {r['elapsed']*1000:.3f} ms  ({ROWS_PER_WRITER/r['elapsed']:,.0f} r/s)")

print(f"\nMedian thread time: {statistics.median(per_thread_times)*1000:.3f} ms")
print(f"Mean thread time  : {statistics.mean(per_thread_times)*1000:.3f} ms")
print(f"Min thread time   : {min(per_thread_times)*1000:.3f} ms")
print(f"Max thread time   : {max(per_thread_times)*1000:.3f} ms")


=== Multi-Writer Benchmark Result ===
Writers (threads): 32
Rows per writer : 1000
Total rows      : 32000
Total elapsed   : 168.395 ms  (0.168395 s)
Overall throughput: 190,030 rows/sec

Worker 0: 83.785 ms  (11,935 r/s)
Worker 1: 93.672 ms  (10,676 r/s)
Worker 2: 79.988 ms  (12,502 r/s)
Worker 3: 82.832 ms  (12,073 r/s)
Worker 4: 100.019 ms  (9,998 r/s)
Worker 5: 89.666 ms  (11,152 r/s)
Worker 6: 92.602 ms  (10,799 r/s)
Worker 7: 82.196 ms  (12,166 r/s)
Worker 8: 98.914 ms  (10,110 r/s)
Worker 9: 89.566 ms  (11,165 r/s)
Worker 10: 93.663 ms  (10,677 r/s)
Worker 11: 91.468 ms  (10,933 r/s)
Worker 12: 98.762 ms  (10,125 r/s)
Worker 13: 91.028 ms  (10,986 r/s)
Worker 14: 91.399 ms  (10,941 r/s)
Worker 15: 93.840 ms  (10,656 r/s)
Worker 16: 97.574 ms  (10,249 r/s)
Worker 17: 89.566 ms  (11,165 r/s)
Worker 18: 84.645 ms  (11,814 r/s)
Worker 19: 93.321 ms  (10,716 r/s)
Worker 20: 82.482 ms  (12,124 r/s)
Worker 21: 96.287 ms  (10,386 r/s)
Worker 22: 88.106 ms  (11,350 r/s)
Worker 23: 93.80

In [12]:
# 6. Cleanup Inserted Rows
# Gather all inserted IDs from all threads
all_inserted_ids = [id for r in all_results for id in r['inserted_ids']]

conn = psycopg2.connect(DSN)
conn.autocommit = False
cur = conn.cursor()

# Get pre-cleanup row count
cur.execute(f'SELECT COUNT(*) FROM {TABLE};')
pre_cleanup_count = cur.fetchone()[0]

# Delete inserted rows
cur.execute(f"DELETE FROM {TABLE} WHERE id = ANY(%s);", (all_inserted_ids,))
deleted = cur.rowcount
conn.commit()

# Get post-cleanup row count
cur.execute(f'SELECT COUNT(*) FROM {TABLE};')
post_cleanup_count = cur.fetchone()[0]

print(f'\nCleanup: deleted {deleted} row(s).')
print(f'Post-cleanup row count: {post_cleanup_count:,}')
assert post_cleanup_count == pre_cleanup_count - deleted, f'Row count mismatch after cleanup!'
print('Assertion passed: pre-existing data intact.')

conn.close()


Cleanup: deleted 32000 row(s).
Post-cleanup row count: 800,011,200
Assertion passed: pre-existing data intact.


## Result Summary

- **Writers (threads):** Number of concurrent writers used in the benchmark.
- **Rows per writer:** Number of rows each thread inserted.
- **Total rows:** Total rows inserted across all threads.
- **Total elapsed:** Wall-clock time for all threads to complete.
- **Overall throughput:** Total rows divided by total elapsed time.
- **Per-thread timings:** Individual timings and throughput for each thread.
- **Cleanup:** All inserted rows are deleted by their IDs to leave pre-existing data intact.

> Adjust `N_WRITERS` and `ROWS_PER_WRITER` to test different concurrency and batch sizes.

---

**Note:** This notebook is designed for benchmarking on a live, indexed table. Use with caution on production systems.